# Gemini API: Basic information extraction

This example notebook shows how Gemini API's JS SDK can be used to extract information from a block of text and return it in defined structure.

In this notebook, the LLM is given a recipe and is asked to extract all the ingredients to create a shopping list. According to best practices, complex tasks will be executed better if divided into separate steps, such as:

1. First, the model will extract all the groceries into a list.
2. Then, you will prompt it to convert this list into a shopping list.

You can find more tips for writing prompts [here](https://ai.google.dev/gemini-api/docs/prompting-intro).


## Setup

### Install the Google GenAI SDK

Install the Google GenAI SDK from [npm](https://www.npmjs.com/package/@google/genai). 

```bash
$ npm install @google/genai
```

### Setup your API key

You can [create](https://aistudio.google.com/app/apikey) your API key using Google AI Studio with a single click.

Remember to treat your API key like a password. Don't accidentally save it in a notebook or source file you later commit to GitHub. In this notebook we will be storing the API key in a `.env` file. You can also set it as an environment variable or use a secret manager. 

Here's how to set it up in a `.env` file:

```bash
$ touch .env
$ echo "GEMINI_API_KEY=<YOUR_API_KEY>" >> .env
```

:::{.callout-tip}

Another option is to set the API key as an environment variable. You can do this in your terminal with the following command:

```bash
$ export GEMINI_API_KEY="<YOUR_API_KEY>"
```
:::

### Load the API key

To load the API key from the `.env` file, we will use the `dotenv` package. This package loads environment variables from a `.env` file into `process.env`. 

```bash
$ npm install dotenv
```

Then, we can load the API key in our code:


In [3]:
const dotenv = require("dotenv") as typeof import("dotenv");

dotenv.config({
  path: "../../.env",
});

const GEMINI_API_KEY = process.env.GEMINI_API_KEY ?? "";
if (!GEMINI_API_KEY) {
  throw new Error("GEMINI_API_KEY is not set in the environment variables");
}
console.log("GEMINI_API_KEY is set in the environment variables");


GEMINI_API_KEY is set in the environment variables


:::{.callout-note}
In our particular case the `.env` is is two directories up from the notebook, hence we need to use `../../` to go up two directories. If the `.env` file is in the same directory as the notebook, you can omit it altogether.

```
│
├── .env
└── examples
    └── prompting
        └── Basic_Information_Extraction.ipynb
```
:::


### Initialize SDK Client

With the new SDK, now you only need to initialize a client with you API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.


In [4]:
const google = require("@google/genai") as typeof import("@google/genai");

const ai = new google.GoogleGenAI({ apiKey: GEMINI_API_KEY });


### Select a model

Now select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](../../quickstarts/Get_started_thinking.ipynb) for more details and in particular learn how to switch the thiking off).


In [5]:
const tslab = require("tslab") as typeof import("tslab");

const MODEL_ID = "gemini-2.5-flash-preview-05-20";


## Example

First, start by extracting all the groceries. To dod this, set the system instructions when defining the model


In [6]:
const GROCERIES_SYSTEM_PROMPT = `
  Your task is to extract to a list all the groceries with its quantities based on the provided recipe.
  Make sure that groceries are in the order of appearance.
`;


Next, the recipe is defined. You will pass the recipe into `generateContent`, and see that the list of groceries was successfully extracted from the input.


In [7]:
const recipe = `
  Step 1:
  Grind 3 garlic cloves, knob of fresh ginger, roughly chopped, 3 spring onions to a paste in a food processor.
  Add 2 tbsp of clear honey, juice from one orange, 1 tbsp of light soy sauce and 2 tbsp of vegetable oil, then blend again.
  Pour the mixture over the cubed chicken from 4 small breast fillets and leave to marnate for at least 1hr.
  Toss in the 20 button mushrooms for the last half an hour so the take on some of the flavour, too.

  Step 2:
  Thread the chicken, 20 cherry tomatoes, mushrooms and 2 large red peppers onto 20 wooden skewers,
  then cook on a griddle pan for 7-8 mins each side or until the chicken is thoroughly cooked and golden brown.
  Turn the kebabs frequently and baste with the marinade from time to time until evenly cooked.
  Arrange on a platter, and eat with your fingers.
`;

const groceries = await ai.models.generateContent({
  model: MODEL_ID,
  contents: recipe,
  config: {
    systemInstruction: GROCERIES_SYSTEM_PROMPT,
  },
});
tslab.display.markdown(groceries.text ?? "");


*   garlic cloves: 3
*   fresh ginger: knob
*   spring onions: 3
*   clear honey: 2 tbsp
*   orange: 1
*   light soy sauce: 1 tbsp
*   vegetable oil: 2 tbsp
*   chicken breast fillets: 4 small
*   button mushrooms: 20
*   cherry tomatoes: 20
*   red peppers: 2 large

The next step is to further format the shopping list based on the ingredients extracted.


In [8]:
const SHOPPING_LIST_SYSTEM_PROMPT = `
  You are given a list of groceries. Complete the following:
  - Organize groceries into categories for easier shopping.
  - List each item one under another with a checkbox [].
`;


Now that you have defined the instructions, you can also decide how you want to format your grocery list. Give the prompt a couple examples, or perform few-shot prompting, so it understands how to format your grocery list.


In [10]:
const shopping_list = await ai.models.generateContent({
  model: MODEL_ID,
  contents: `
      LIST: 3 tomatoes, 1 turkey, 4 tomatoes
      OUTPUT:
      ## VEGETABLES
      - [ ] 7 tomatoes
      ## MEAT
      - [ ] 1 turkey
    
      LIST: ${groceries.text}
      OUTPUT:
    `,
  config: {
    systemInstruction: SHOPPING_LIST_SYSTEM_PROMPT,
  },
});
tslab.display.markdown(shopping_list.text ?? "");


## PRODUCE
- [ ] 3 garlic cloves
- [ ] 1 knob fresh ginger
- [ ] 3 spring onions
- [ ] 1 orange
- [ ] 20 button mushrooms
- [ ] 20 cherry tomatoes
- [ ] 2 large red peppers

## MEAT
- [ ] 4 small chicken breast fillets

## PANTRY
- [ ] 2 tbsp clear honey
- [ ] 1 tbsp light soy sauce
- [ ] 2 tbsp vegetable oil

## Next steps

Be sure to explore other examples of prompting in the repository. Try creating your own prompts for information extraction or adapt the ones provided in the notebook.
